In [23]:
# ============================================================
# FLEX + BISON : Arithmetic Expression Validator
# Operators: +, -, *, /
# ============================================================

# Install Flex, Bison and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# 1. Create FLEX file : art_expr.l
# ============================================================

lexer_code = r'''
%{
#include <stdio.h>
#include "art_expr.tab.h"
%}

%%
[a-zA-Z][0-9a-zA-Z]*    { return ID; }
[0-9]+                  { return DIG; }
[ \t]+                  { /* ignore spaces */ }
\n                      { return 0; }
.                       { return yytext[0]; }
%%

int yywrap()
{
    return 1;
}
'''

with open("art_expr.l", "w") as f:
    f.write(lexer_code)


# ============================================================
# 2. Create BISON file : art_expr.y
# ============================================================

parser_code = r'''
%{
#include <stdio.h>
#include <stdlib.h>

int yylex();
int yyerror(const char *s);
%}

%token ID DIG

%left '+' '-'
%left '*' '/'

%right UMINUS

%%

stmt:
      expn
      ;

expn:
      expn '+' expn
    | expn '-' expn
    | expn '*' expn
    | expn '/' expn
    | '-' expn %prec UMINUS
    | '(' expn ')'
    | DIG
    | ID
    ;

%%

int main()
{
    printf("Enter the Expression:\n");

    if (yyparse() == 0)
    {
        printf("Valid Expression\n");
    }

    return 0;
}

int yyerror(const char *s)
{
    printf("Invalid Expression\n");
    exit(0);
}
'''

with open("art_expr.y", "w") as f:
    f.write(parser_code)


# ============================================================
# 3. Generate parser using BISON
# ============================================================

!bison -d art_expr.y


# ============================================================
# 4. Generate lexical analyzer using FLEX
# ============================================================

!flex art_expr.l


# ============================================================
# 5. Compile FLEX + BISON
# ============================================================

!gcc lex.yy.c art_expr.tab.c -o art_expr -lfl


# ============================================================
# 6. Run the program with VALID expression
# ============================================================

print("\n========== VALID EXPRESSION TEST ==========\n")

with open("input.txt", "w") as f:
    f.write("a+b*c-d/e\n")

!./art_expr < input.txt


# ============================================================
# 7. Run the program with INVALID expression
# ============================================================

print("\n========== INVALID EXPRESSION TEST ==========\n")

with open("input.txt", "w") as f:
    f.write("a=b\n")

!./art_expr < input.txt

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package bison.
(Reading database ... 118333 files and directories currently installed.)
Preparing to unpack .../bison_2%3a3.8.2+dfsg-1build1_amd64.deb ...
Unpacking bison (2:3.8.2+dfsg-1build1) ...
Setting up bison (2:3.8.2+dfsg-1build1) ...
update-alternatives: using /usr/bin/bison.yacc to provide /usr/bin/yacc (yacc) in auto mode
Processing triggers for man-db (2.10.2-1) ...

========== VALID EXPRESSION TEST ==========

Enter the Expression:
Valid Expression

========== INVALID EXPRESSION TEST ==========

Enter the Expression:
Invalid Expression
